In [65]:
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler



In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [67]:
df = pd.read_csv('data/preprocessed_data.csv')


In [68]:
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

In [69]:
num_feature = X.select_dtypes(exclude='object').columns

In [70]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

In [71]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_feature)
    ]
)


In [72]:
from sklearn.model_selection import train_test_split

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [74]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(),
    "SVC": SVC(),
    "MLPClassifier": MLPClassifier(max_iter=1000)
}

In [75]:
base_results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    
    train_acc = pipe.score(X_train, y_train)
    test_acc = pipe.score(X_test, y_test)
    
    base_results[name] = (train_acc, test_acc)
    
    print(f"{name}: Train={train_acc:.4f}, Test={test_acc:.4f}")

LogisticRegression: Train=0.9868, Test=0.9737
RandomForest: Train=1.0000, Test=0.9649
SVC: Train=0.9868, Test=0.9825
MLPClassifier: Train=1.0000, Test=0.9649


In [76]:
param_grids = {
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10],
        "model__solver": ["liblinear", "lbfgs"]
    },

    "RandomForest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5]
    },

    "SVC": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    },

    "MLPClassifier": {
        "model__hidden_layer_sizes": [(50,), (100,), (100,50)],
        "model__activation": ["relu", "tanh"],
        "model__learning_rate_init": [0.001, 0.01]
    }
}

In [77]:
best_models = {}
tuned_results = {}

In [78]:
for name, model in models.items():
    
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    grid = GridSearchCV(
        pipe,
        param_grids[name],
        cv=3,
        scoring='accuracy',
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_models[name] = grid.best_estimator_
    
    train_acc = grid.best_estimator_.score(X_train, y_train)
    test_acc = grid.best_estimator_.score(X_test, y_test)
    
    tuned_results[name] = (train_acc, test_acc, grid.best_params_)
    
    print(f"\n{name}")
    print(f"Best Params: {grid.best_params_}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")


LogisticRegression
Best Params: {'model__C': 0.1, 'model__solver': 'liblinear'}
Train Accuracy: 0.9824
Test Accuracy: 0.9912

RandomForest
Best Params: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Train Accuracy: 0.9956
Test Accuracy: 0.9649

SVC
Best Params: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Train Accuracy: 0.9868
Test Accuracy: 0.9825

MLPClassifier
Best Params: {'model__activation': 'relu', 'model__hidden_layer_sizes': (100,), 'model__learning_rate_init': 0.001}
Train Accuracy: 1.0000
Test Accuracy: 0.9737


In [79]:
best_model_name = max(tuned_results, key=lambda x: tuned_results[x][1])
best_train, best_test, best_params = tuned_results[best_model_name]

print("\n========= BEST MODEL =========")
print(f"Best Model: {best_model_name}")
print(f"Train Accuracy: {best_train:.4f}")
print(f"Test Accuracy: {best_test:.4f}")
print(f"Best Parameters: {best_params}")


========= BEST MODEL =========
Best Model: LogisticRegression
Train Accuracy: 0.9824
Test Accuracy: 0.9912
Best Parameters: {'model__C': 0.1, 'model__solver': 'liblinear'}


Required comparison